# 🏭 AI-Enabled Assets Performance & Predictive Maintenance Platform

This notebook allows you to run the full **Industrial Risk AI** platform directly in Google Colab without installing anything locally.

**Note:** To keep the dashboard running, leave this browser tab open.

### Step 1: 🚀 Setup Environment
Run this cell to clone the repository and install all required Python libraries.

In [ ]:
import os
import subprocess
import time

# 1. Clone Repository
REPO_URL = "https://github.com/lmudu2/industrial-risk-ai.git"
REPO_DIR = "industrial-risk-ai"

if not os.path.exists(REPO_DIR):
    print(f"Cloning {REPO_URL}...")
    !git clone {REPO_URL}

os.chdir(f"/content/{REPO_DIR}")

# 2. Install Dependencies
print("Installing dependencies (this may take 1-2 minutes)...")
!pip install -q -r requirements.txt

print("✅ Setup Complete!")

### Step 2: 📊 Generate Industrial Database
Run this cell to generate the synthesized SQLite database containing assets, work orders, sensors, and risk scores.

In [ ]:
if not os.path.exists("backend/eam_database.db"):
    print("Database not found. Generating real-world industrial data (3-5 minutes)...")
    !python data/generate_data.py
else:
    print("✅ Database already exists. Skipping generation.")

### Step 3: ⚡ Start the Platform
This cell starts the **FastAPI Backend** and the **Streamlit Frontend**.

We use a specialized tunnel to prevent "Module Script Failed" errors common in Colab.

In [ ]:
from google.colab import userdata
import urllib.request

# 0. Load Secrets
try:
    os.environ["GROQ_API_KEY"] = userdata.get("GROQ_API_KEY")
    print("✅ GROQ_API_KEY loaded!")
except:
    print("⚠️ WARNING: GROQ_API_KEY not found in Colab Secrets.")

# 1. Start FastAPI Backend
print("Starting Backend API...")
subprocess.Popen(["uvicorn", "backend.main:app", "--host", "0.0.0.0", "--port", "8000"])
time.sleep(2)

# 2. Start Streamlit Frontend
print("Starting Streamlit Dashboard...")
subprocess.Popen(["streamlit", "run", "frontend/app.py", "--server.port", "8501", "--server.address", "0.0.0.0"])
time.sleep(5)

# 3. Start Tunnel (Pinggy - Zero Auth, Zero Config, High Compatibility)
print("Opening Secure Tunnel...\n")
!ssh -o StrictHostKeyChecking=no -p 443 -R 80:localhost:8501 a.pinggy.io